# Values behind the data figures (Figs. 2 and 5)

Collects the numbers plotted in Figs. 2 to 5 so the figures can be redrawn without changing any result.

* Fig. 5 (distance distributions): Mahalanobis distances recomputed from the cached embeddings, as in the
  baseline-comparison notebook. This computation is deterministic; the check prints the Tufts refusal rates.
* Fig. 2 (reliability diagrams): test-set probabilities of the saved DenseNet-121, the network selected by
  validation macro-AUC. Inference only.

Each section prints a check against the manuscript values. Output: `_results_figures/figures_data.json` in the DENTEX
folder. Runtime: one to two minutes.

## 1. Paths

In [ ]:
DENTAL = "/path/to/Dental"
import os, json, random, datetime
import numpy as np
DATA_DIR = os.path.join(DENTAL, "DENTEX/training_data/quadrant-enumeration-disease")
ANN_PATH = os.path.join(DATA_DIR, "train_quadrant_enumeration_disease.json")
IMG_DIR  = os.path.join(DATA_DIR, "xrays")
OPT1     = os.path.join(DATA_DIR, "_results_option1")
CACHE    = os.path.join(OPT1, "emb_cache")
CKPT     = os.path.join(DATA_DIR, "_results", "checkpoints", "densenet121_seed42.pt")
OUT_DIR  = os.path.join(DATA_DIR, "_results_figures"); os.makedirs(OUT_DIR, exist_ok=True)
for p in (ANN_PATH, IMG_DIR, OPT1, CACHE, CKPT):
    print("found" if os.path.exists(p) else "MISSING", p)
OUT = {"version": "figures_data_v1", "timestamp": datetime.datetime.now().isoformat(timespec="seconds")}
def same(a, b, tol): return "OK" if abs(a - b) <= tol else "DIFFERENT"

## 2. Teeth, labels and the seed-42 split (as in notebooks 01 and 03)

In [ ]:
coco = json.load(open(ANN_PATH))
cats3 = sorted(coco["categories_3"], key=lambda c: c["id"])
DX_NAMES = [c["name"] for c in cats3]; NUM_DX = len(DX_NAMES)
id2idx = {c["id"]: i for i, c in enumerate(cats3)}
img_file = {im["id"]: im["file_name"] for im in coco["images"]}
tooth = {}
for a in coco["annotations"]:
    key = (a["image_id"], a.get("category_id_1", -1), a.get("category_id_2", -1))
    if key not in tooth:
        tooth[key] = {"image_id": a["image_id"], "file": img_file[a["image_id"]], "bbox": list(a["bbox"]),
                      "y": np.zeros(NUM_DX, np.float32)}
    tooth[key]["y"][id2idx[a["category_id_3"]]] = 1.0
    bx, by, bw, bh = tooth[key]["bbox"]; ax, ay, aw, ah = a["bbox"]
    x0, y0 = min(bx, ax), min(by, ay); x1, y1 = max(bx + bw, ax + aw), max(by + bh, ay + ah)
    tooth[key]["bbox"] = [x0, y0, x1 - x0, y1 - y0]
teeth = list(tooth.values())
Y = np.stack([t["y"] for t in teeth])

def grouped_split(n_seed, vf=0.15, tf=0.15):
    ids = sorted({t["image_id"] for t in teeth}); rng = np.random.RandomState(n_seed); rng.shuffle(ids)
    n = len(ids); nte = int(n * tf); nva = int(n * vf)
    te = set(ids[:nte]); va = set(ids[nte:nte + nva]); a, b, c = [], [], []
    for i, t in enumerate(teeth):
        (c if t["image_id"] in te else b if t["image_id"] in va else a).append(i)
    return np.array(a), np.array(b), np.array(c)
tr, va, te = grouped_split(42)
print(len(teeth), "teeth | split", len(tr), len(va), len(te), "| expected 3456 teeth, 2416 / 511 / 529")
OUT["classes"] = DX_NAMES

## 3. Fig. 5: Mahalanobis distances of DENTEX test and Tufts crops (recomputed from the caches)

In [ ]:
def maha_fit(X):
    mu = X.mean(0); ci = np.linalg.inv(np.cov(X.T) + np.eye(X.shape[1]) * 1e-2)
    return lambda Z: np.einsum("ni,ij,nj->n", Z - mu, ci, Z - mu)
OUT["fig5"] = {}
for n in ("dental_vitl", "biomedclip"):
    E = np.load(os.path.join(CACHE, f"emb_{n}.npy")).astype(np.float32)
    Tu = np.load(os.path.join(CACHE, f"tufts_box_emb_{n}.npy")).astype(np.float32)
    assert len(E) == len(teeth), (n, E.shape)
    m = maha_fit(E[tr]); d_te, d_va, d_tu = m(E[te]), m(E[va]), m(Tu)
    t_test, t_val = float(np.quantile(d_te, 0.95)), float(np.quantile(d_va, 0.95))
    OUT["fig5"][n] = {"dentex_test": np.round(d_te, 3).tolist(), "tufts": np.round(d_tu, 3).tolist(),
                      "threshold_test_p95": t_test, "threshold_val_p95": t_val,
                      "tufts_refused_test_p95": float((d_tu > t_test).mean()),
                      "tufts_refused_val_p95": float((d_tu > t_val).mean()),
                      "dentex_test_refused_val_p95": float((d_te > t_val).mean())}
    r = OUT["fig5"][n]
    print(f"{n}: Tufts refused {r['tufts_refused_test_p95']:.1%} (test p95) / {r['tufts_refused_val_p95']:.1%} (val p95)"
          f" | DENTEX test refused at val p95 {r['dentex_test_refused_val_p95']:.1%} | n Tufts {len(d_tu)}")
print("manuscript: Dental ViT-L 84.7% / 87.9%, DENTEX 8.7%; BiomedCLIP 78.2% / 79.3%, DENTEX 5.3%")

## 4. Fig. 2: test-set probabilities of the saved DenseNet-121 (inference only)

In [ ]:
import torch, torch.nn as nn, torchvision.transforms as T
from torchvision import models
from PIL import Image
from sklearn.metrics import roc_auc_score
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
NORM = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
eval_tf = T.Compose([T.Grayscale(3), T.Resize((224, 224)), T.ToTensor(), NORM])
def crop_tooth(img, bbox, pad=0.15):
    W, H = img.size; x, y, w, h = bbox; px, py = w * pad, h * pad
    x0 = max(0, int(x - px)); y0 = max(0, int(y - py)); x1 = min(W, int(x + w + px)); y1 = min(H, int(y + h + py))
    return img if (x1 <= x0 or y1 <= y0) else img.crop((x0, y0, x1, y1))
net = models.densenet121(weights=None); net.classifier = nn.Linear(net.classifier.in_features, NUM_DX)
net.load_state_dict(torch.load(CKPT, map_location="cpu")); net = net.to(DEVICE).eval()
cache, P = {}, []
with torch.no_grad():
    for k in range(0, len(te), 32):
        xs = []
        for i in te[k:k + 32]:
            t = teeth[i]
            if t["file"] not in cache:
                cache[t["file"]] = Image.open(os.path.join(IMG_DIR, os.path.basename(t["file"]))).convert("L")
            xs.append(eval_tf(crop_tooth(cache[t["file"]], t["bbox"])))
        P.append(torch.sigmoid(net(torch.stack(xs).to(DEVICE))).cpu().numpy())
P = np.concatenate(P); Yt = Y[te]
def ece(y, p, nb=10):
    b = np.linspace(0, 1, nb + 1); e = 0.0
    for lo, hi in zip(b[:-1], b[1:]):
        m = (p > lo) & (p <= hi)
        if m.sum(): e += m.sum() / len(p) * abs(p[m].mean() - y[m].mean())
    return e
auc = np.mean([roc_auc_score(Yt[:, c], P[:, c]) for c in range(NUM_DX)])
auc = float(auc); print(f"macro-AUC {auc:.4f} {same(auc, 0.9272, 2e-3)} (validation_selected_v1.json: 0.9272)")
print("ECE:", {DX_NAMES[c]: round(float(ece(Yt[:, c], P[:, c])), 4) for c in range(NUM_DX)},
      "| expected 0.0201 / 0.0935 / 0.0396 / 0.0787")
OUT["fig2"] = {"network": "densenet121", "y": Yt.astype(int).tolist(), "p": np.round(P, 6).tolist()}

## 5. Save

In [ ]:
path = os.path.join(OUT_DIR, "figures_data.json")
json.dump(OUT, open(path, "w"))
print("Saved", path, f"({os.path.getsize(path) / 1e6:.1f} MB)")